# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliHamza-lab/Ml_intership/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My lane is **Refresh / Content Opportunity Scoring**.

*   **ML Task Type**: **Ranking and Scoring** (supported by Binary Classification).
*   **Why?**: Although we train a binary classification model underneath (to predict whether a page is declining or not), the end deliverable is a prioritized list (a ranked queue) of pages for a content editor to refresh. We use the model's output probabilities as a continuous "refresh opportunity score" to rank all candidate pages in descending order. This makes it fundamentally a ranking/scoring task, matching the operational constraint of limited human capacity.

In [1]:
print("Task Type: Scoring & Ranking (Binary Classification proxy)")
print("Output: Continuous probability score [0.0, 1.0] used to rank candidates in a queue.")

Task Type: Scoring & Ranking (Binary Classification proxy)
Output: Continuous probability score [0.0, 1.0] used to rank candidates in a queue.


## 2. Target or proxy

*   **What is predicted**: A binary proxy label: `is_declining_label` (where `1` means the page is declining and `0` means it is stable/growing).
*   **Label Origin**: In this starter phase, it is a **rule-derived proxy label** calculated directly from the observed data column `trend_direction` being `"down"` (`is_declining_label = trend_direction == 'down'`).
*   **Capstone Evolution**: For the full capstone, this will evolve into a true **future-window observed label** (e.g., predicting whether a page's organic traffic drops by >=20% in a future 30-day window, using only features computed from the prior 90-day window), preventing any temporal leakage.

In [2]:
# Find repo root and load the dataset
import os, pandas as pd, numpy as np
for _ in range(5):
    if os.path.isdir("data/raw"): break
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"Target distribution (is_declining_label):")
print(df["is_declining_label"].value_counts(normalize=True).round(3).to_string())

ModuleNotFoundError: No module named 'pandas'

## 3. Success metric

*   **Defensible Metric**: **Precision@K** (specifically `Precision@50`).
*   **Why this metric?**: Content editors have fixed operational capacity (e.g., they can only rewrite or refresh a few pages per week, say 50 pages). If our top recommendations are wrong, we waste expensive writing/editing hours. Precision@50 measures the fraction of the top 50 flagged pages that are actually declining. We prioritize high precision at the top of the queue (low false positives) over broad recall, as we do not need to find every declining page, but we must be highly confident in the ones we recommend to fix first.

In [ ]:
# Define the precision_at_k function to demonstrate the metric calculation
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Create a dummy score (e.g. rank by days_since_last_update) to show precision@50
dummy_precision = precision_at_k(df["days_since_last_update"], df["is_declining_label"], 50)
print(f"Dummy baseline (stale-only) Precision@50: {dummy_precision:.3f}")

Dummy baseline (stale-only) Precision@50: 0.540


## 4. The unit of analysis, as a real dataframe

*   **Unit of Analysis**: A **single content item (page) for a given client** (Grain: `client_id` + `content_id`).
*   **Below is the slice representation**: We display the key features (impressions, CTR, average position, last update age, content age) and our target label, showing that each row represents a unique page in the context of a client's site.

In [ ]:
# Show the unit of analysis as a clean slice
columns_to_show = [
    "client_id", "content_id", "impressions_90d", "ctr", 
    "avg_position", "days_since_last_update", "content_age_days", "is_declining_label"
]
print("DataFrame Slice (Unit of Analysis):")
df_slice = df[columns_to_show].head(5)
display(df_slice)

DataFrame Slice (Unit of Analysis):


,client_id,content_id,impressions_90d,ctr,avg_position,days_since_last_update,content_age_days,is_declining_label
0,client_f369cb89fc,content_304f48230142,3803,0.76,10.6,20,187,1
1,client_4e07408562,content_a1fb4e703a9e,15320,0.05,20.3,25,445,1
2,client_7f2253d7e2,content_9aa793d4d895,12581,0.09,36.5,20,141,1
3,client_19581e27de,content_331d6c4de07b,11751,0.49,6.2,22,463,0
4,client_3fdba35f04,content_d99b7a2d90ca,19140,0.13,44.0,14,263,1


## 5. Why ML beats a fixed rule here

*   **Multi-Dimensional Interactions**: A simple hand-written rule like `stale AND visible` relies on hard, arbitrary thresholds (e.g., `days_since_last_update >= 180` and `impressions_90d >= 500`). It ignores the non-linear interaction of other critical signals like a collapsing CTR, high CPC (indicating high commercial intent), low GA4 engagement/scroll rates, and search volume.
*   **Continuous Ranking vs. Binary Flags**: If-statements classify pages into coarse buckets, leading to massive ties (e.g., hundreds of pages matching the exact same rule). ML calculates a continuous, fine-grained probability score that allows us to build a smooth, prioritized queue.
*   **Signal Complexity**: Features like CTR decay behave differently depending on the average position (as shown by the CTR cliff). A tree-based ML model automatically learns these threshold interactions (e.g. splitting CTR based on average position tiers), which would require thousands of manual nested if/else statements to match by hand.

In [ ]:
# Demonstrate feature correlations to show the complex, multi-variable signal mix
correlation_matrix = df[["days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]].corrwith(df["is_declining_label"])
print("Correlation of key features with target label:")
print(correlation_matrix.round(3).to_string())

Correlation of key features with target label:
days_since_last_update    0.081
impressions_90d          -0.018
avg_position             -0.029
ctr                      -0.062
word_count                0.090


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.